# Patient Treatment Journey Experience Analyzer

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Track sentiment across journey stages** using `AI_SENTIMENT()`
2. **Extract pain points** with `AI_COMPLETE()` and LLM reasoning
3. **Summarize journey patterns** using `AI_SUMMARIZE()`
4. **Classify patient risk** with `AI_CLASSIFY()`
5. **Build journey funnels** with SQL window functions
6. **Create intervention dashboards** with Streamlit

---

## What You'll Build

A **patient journey intelligence system** analyzing treatment experiences:
- **6-stage journey tracking**: Diagnosis → Prescription → Onboarding → Treatment → Refills → Outcomes
- **Sentiment scoring** at each touchpoint (-1 to +1 scale)
- **Pain point extraction** from negative feedback using LLM
- **Journey stage summaries** for intervention planning
- **Risk classification** (Alto/Medio/Bajo drop-off likelihood)

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 14-16 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SENTIMENT()` - Journey sentiment tracking
- `AI_COMPLETE()` - Pain point extraction
- `AI_SUMMARIZE()` - Stage summaries
- `AI_CLASSIFY()` - Risk segmentation
- Window functions - Journey sequencing
- `CASE` statements - Stage classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `PATIENT_JOURNEY_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Patient Journey Intelligence

Understanding where patients drop off is critical for:
1. **Improving patient outcomes** - Identify and address barriers early
2. **Optimizing support programs** - Target interventions at key stages
3. **Reducing drop-off rates** - Proactive outreach before patients quit

### Why This Matters

Cortex AI enables **automated sentiment tracking** across thousands of patient touchpoints, **LLM-powered theme extraction** from unstructured feedback, and **predictive risk scoring** to identify at-risk patients.

In [ ]:
CREATE DATABASE IF NOT EXISTS PATIENT_JOURNEY_DB;
CREATE SCHEMA IF NOT EXISTS PATIENT_JOURNEY_DB.PUBLIC;
USE SCHEMA PATIENT_JOURNEY_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as database_name, CURRENT_SCHEMA() as schema_name, CURRENT_WAREHOUSE() as warehouse_name;

---

## Paso 2: Define Journey Stages & Tables

### Patient Journey Stages

We'll track patients through **6 critical stages**:

1. **Diagnosis** - Initial diagnosis, doctor consultation
2. **Prescription** - Prescription written, insurance authorization
3. **Onboarding** - First fill, patient education
4. **Active Treatment** - Regular dosing, side effect management
5. **Refills** - Ongoing prescriptions, adherence tracking
6. **Outcomes** - Clinical results, A1C improvement

### Data Schema

- `patients` - Patient demographics and cohort
- `journey_interactions` - Every touchpoint (app, call, clinic, pharmacy)
- `patient_feedback` - Unstructured feedback text
- `journey_completion` - Final outcome (completed vs. dropped off)

In [ ]:
CREATE OR REPLACE TABLE patients (
    patient_id VARCHAR PRIMARY KEY,
    age INTEGER,
    gender VARCHAR(10),
    diagnosis VARCHAR(50),
    insurance_type VARCHAR(30),
    enrollment_date DATE,
    cohort VARCHAR(20)
);

CREATE OR REPLACE TABLE journey_interactions (
    interaction_id VARCHAR PRIMARY KEY,
    patient_id VARCHAR,
    journey_stage VARCHAR(20),
    touchpoint_type VARCHAR(30),
    interaction_date DATE,
    interaction_text VARCHAR(500),
    stage_order INTEGER
);

CREATE OR REPLACE TABLE patient_feedback (
    feedback_id VARCHAR PRIMARY KEY,
    patient_id VARCHAR,
    journey_stage VARCHAR(20),
    feedback_date DATE,
    feedback_text VARCHAR(1000),
    is_negative BOOLEAN
);

CREATE OR REPLACE TABLE journey_completion (
    patient_id VARCHAR PRIMARY KEY,
    completed_journey BOOLEAN,
    drop_off_stage VARCHAR(20),
    total_touchpoints INTEGER,
    journey_duration_days INTEGER,
    final_outcome VARCHAR(50)
);

SELECT 'Tables created successfully!' as status;


In [ ]:
INSERT INTO patients
SELECT
    'P-' || LPAD(SEQ4(), 6, '0') as patient_id,
    UNIFORM(35, 75, RANDOM()) as age,
    CASE WHEN UNIFORM(0, 1, RANDOM()) = 0 THEN 'Male' ELSE 'Female' END as gender,
    CASE WHEN UNIFORM(1, 10, RANDOM()) <= 7 THEN 'Atrial Fibrillation' ELSE 'Venous Thromboembolism' END as diagnosis,
    CASE 
        WHEN UNIFORM(1, 10, RANDOM()) <= 6 THEN 'Commercial'
        WHEN UNIFORM(1, 10, RANDOM()) <= 9 THEN 'Medicare'
        ELSE 'Medicaid'
    END as insurance_type,
    DATEADD(day, -UNIFORM(1, 365, RANDOM()), CURRENT_DATE()) as enrollment_date,
    CASE 
        WHEN UNIFORM(1, 10, RANDOM()) <= 4 THEN 'High Engagement'
        WHEN UNIFORM(1, 10, RANDOM()) <= 8 THEN 'Medium Engagement'
        ELSE 'At Risk'
    END as cohort
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

SELECT COUNT(*) as total_patients, cohort, COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () as percentage
FROM patients
GROUP BY cohort
ORDER BY percentage DESC;


---

## Paso 4: Generar Journey Interactions

### Touchpoint Types

Each patient has **6-12 touchpoints** across their journey:
- **App Check-In** - Mobile app usage (daily logs, reminders)
- **Phone Call** - Support line, nurse coaching
- **Clinic Visit** - In-person appointments with HCP
- **Pharmacy Interaction** - Refills, questions to pharmacist
- **Email/SMS** - Automated reminders, educational content

### Stage Progression

Patients move through stages sequentially. Some drop off before completion.

In [ ]:
INSERT INTO journey_interactions
WITH stages AS (
    SELECT * FROM (VALUES
        ('Diagnosis', 1),
        ('Prescription', 2),
        ('Onboarding', 3),
        ('Active Treatment', 4),
        ('Refills', 5),
        ('Outcomes', 6)
    ) AS t(stage_name, stage_order)
),
patient_stages AS (
    SELECT 
        p.patient_id,
        s.stage_name,
        s.stage_order,
        DATEADD(day, (s.stage_order - 1) * 30 + UNIFORM(0, 15, RANDOM()), p.enrollment_date) as stage_date
    FROM patients p
    CROSS JOIN stages s
    WHERE s.stage_order <= CASE 
        WHEN p.cohort = 'At Risk' THEN UNIFORM(2, 4, RANDOM())
        WHEN p.cohort = 'Medium Engagement' THEN UNIFORM(4, 6, RANDOM())
        ELSE 6
    END
),
touchpoint_gen AS (
    SELECT 
        ps.patient_id,
        ps.stage_name,
        ps.stage_order,
        ps.stage_date,
        ROW_NUMBER() OVER (ORDER BY ps.patient_id, ps.stage_order, RANDOM()) as touch_seq
    FROM patient_stages ps,
    TABLE(GENERATOR(ROWCOUNT => 2))
    WHERE UNIFORM(1, 100, RANDOM()) <= 60
)
SELECT
    'INT-' || LPAD(touch_seq, 8, '0') as interaction_id,
    patient_id,
    stage_name as journey_stage,
    CASE UNIFORM(1, 5, RANDOM())
        WHEN 1 THEN 'App Check-In'
        WHEN 2 THEN 'Phone Call'
        WHEN 3 THEN 'Clinic Visit'
        WHEN 4 THEN 'Pharmacy Interaction'
        ELSE 'Email/SMS'
    END as touchpoint_type,
    stage_date as interaction_date,
    'Interaction at ' || stage_name || ' stage' as interaction_text,
    stage_order
FROM touchpoint_gen;

SELECT 
    journey_stage,
    COUNT(*) as total_interactions,
    COUNT(DISTINCT patient_id) as unique_patients
FROM journey_interactions
GROUP BY journey_stage, stage_order
ORDER BY stage_order;


---

## Paso 5: Generar Patient Feedback

### Feedback Distribution

**3,000+ feedback entries** from patients at various stages:
- **Positive feedback** (60%) - "Feeling great!", "A1C improving!"
- **Neutral feedback** (20%) - "Adjusting to routine", "Waiting for results"
- **Negative feedback** (20%) - Side effects, cost concerns, access barriers

### Purpose

`CORTEX.SENTIMENT()` will analyze these feedback entries, and `CORTEX.COMPLETE()` will extract specific pain points from negative feedback.

In [ ]:
INSERT INTO patient_feedback
WITH feedback_templates AS (
    SELECT * FROM (VALUES
        ('Positive', 'Feeling much better since starting treatment! My energy is back.'),
        ('Positive', 'The app reminders are really helpful for staying on track.'),
        ('Positive', 'My A1C dropped from 9.2 to 7.1 - so grateful!'),
        ('Positive', 'The support team answered all my questions patiently.'),
        ('Neutral', 'Still adjusting to the medication routine.'),
        ('Neutral', 'Waiting to see results at my next appointment.'),
        ('Neutral', 'The process is manageable so far.'),
        ('Negative', 'The cost is really high even with insurance - struggling to afford refills.'),
        ('Negative', 'Experiencing nausea and headaches as side effects.'),
        ('Negative', 'Pharmacy keeps running out of stock - very frustrating.'),
        ('Negative', 'The prior authorization took 3 weeks and I ran out of medication.'),
        ('Negative', 'No one explained the side effects properly.')
    ) AS t(sentiment_type, template_text)
)
SELECT
    'FB-' || LPAD(SEQ4(), 8, '0') as feedback_id,
    ji.patient_id,
    ji.journey_stage,
    ji.interaction_date as feedback_date,
    ft.template_text as feedback_text,
    CASE WHEN ft.sentiment_type = 'Negative' THEN TRUE ELSE FALSE END as is_negative
FROM journey_interactions ji
CROSS JOIN feedback_templates ft
WHERE UNIFORM(1, 10, RANDOM()) <= CASE 
    WHEN ft.sentiment_type = 'Positive' THEN 6
    WHEN ft.sentiment_type = 'Neutral' THEN 8
    ELSE 10
END
AND UNIFORM(1, 3, RANDOM()) = 1
LIMIT 3000;

SELECT 
    is_negative,
    COUNT(*) as feedback_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as percentage
FROM patient_feedback
GROUP BY is_negative;


---

## Paso 6: Track Journey Completion

### Completion Logic

Patients who reached **Outcomes** stage = **Completed Journey**

Otherwise, they dropped off at an earlier stage.

### Drop-Off Analysis

This table enables:
- Identification of where patients quit
- Calculation of completion rates by cohort
- Prioritization of stages for intervention

In [ ]:
INSERT INTO journey_completion
WITH patient_progress AS (
    SELECT 
        ji.patient_id,
        MAX(ji.stage_order) as max_stage_reached,
        MAX(CASE WHEN ji.journey_stage = 'Outcomes' THEN 1 ELSE 0 END) as reached_outcomes,
        MAX(ji.journey_stage) as last_stage,
        COUNT(*) as total_touchpoints,
        DATEDIFF(day, MIN(ji.interaction_date), MAX(ji.interaction_date)) as journey_duration_days
    FROM journey_interactions ji
    GROUP BY ji.patient_id
)
SELECT
    pp.patient_id,
    CASE WHEN pp.reached_outcomes = 1 THEN TRUE ELSE FALSE END as completed_journey,
    CASE WHEN pp.reached_outcomes = 0 THEN pp.last_stage ELSE NULL END as drop_off_stage,
    pp.total_touchpoints,
    pp.journey_duration_days,
    CASE 
        WHEN pp.reached_outcomes = 1 THEN 'Completed Treatment'
        ELSE 'Dropped Off at ' || pp.last_stage
    END as final_outcome
FROM patient_progress pp;

SELECT 
    completed_journey,
    COUNT(*) as patient_count,
    ROUND(AVG(journey_duration_days), 0) as avg_duration_days,
    ROUND(AVG(total_touchpoints), 1) as avg_touchpoints
FROM journey_completion
GROUP BY completed_journey;


---

## Paso 7: Sentiment Analysis with CORTEX.SENTIMENT()

### Qué Hace Esto

`AI_SENTIMENT()` analyzes feedback text and returns a score:
- **-1 to -0.5**: Altoly negative
- **-0.5 to 0**: Slightly negative
- **0 to +0.5**: Slightly positive
- **+0.5 to +1**: Altoly positive

### Why This Matters

Automated sentiment scoring enables:
1. **Real-time monitoring** of patient satisfaction trends
2. **Stage-specific analysis** (which stages have negative sentiment?)
3. **Trigger alerts** for patients with declining sentiment

In [ ]:
CREATE OR REPLACE TABLE feedback_sentiment AS
SELECT
    pf.feedback_id,
    pf.patient_id,
    pf.journey_stage,
    pf.feedback_text,
    -- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT (Use Case 28 pattern)
    CASE AI_SENTIMENT(pf.feedback_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as sentiment_score,
    CASE AI_SENTIMENT(pf.feedback_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 'Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Very Negative'
    END as sentiment_category
FROM patient_feedback pf
LIMIT 500;

SELECT 
    journey_stage,
    sentiment_category,
    COUNT(*) as feedback_count,
    ROUND(AVG(sentiment_score), 3) as avg_sentiment
FROM feedback_sentiment
GROUP BY journey_stage, sentiment_category
ORDER BY journey_stage, sentiment_category;

---

## Paso 8: Extract Pain Points with CORTEX.COMPLETE()

### Qué Hace Esto

`AI_COMPLETE()` uses LLM reasoning to:
1. **Read negative feedback** text
2. **Classify the barrier type** (Cost, Side Effects, Access, etc.)
3. **Extract specific issues** for intervention planning

### LLM Prompt Strategy

We provide the LLM with:
- **Context**: "You are analyzing patient feedback"
- **Task**: "Classify this barrier into one category"
- **Options**: Cost/Insurance, Side Effects, Access Barriers, Process Complexity, Lack of Information

In [ ]:
CREATE OR REPLACE TABLE pain_point_analysis AS
SELECT
    fs.feedback_id,
    fs.patient_id,
    fs.journey_stage,
    fs.feedback_text,
    fs.sentiment_score,
    AI_COMPLETE('mistral-large2',
        'You are analyzing patient treatment barriers. Classify this feedback into ONE category: Cost/Insurance, Side Effects, Access Barriers, Process Complexity, or Lack of Information. Only return the category name. Feedback: ' || fs.feedback_text
    ) as barrier_category
FROM feedback_sentiment fs
WHERE fs.sentiment_score < 0
LIMIT 200;

SELECT 
    barrier_category,
    COUNT(*) as occurrence_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as percentage,
    LISTAGG(DISTINCT journey_stage, ', ') WITHIN GROUP (ORDER BY journey_stage) as affected_stages
FROM pain_point_analysis
GROUP BY barrier_category
ORDER BY occurrence_count DESC;


---

## Paso 9: Summarize Journey Stages with CORTEX.SUMMARIZE()

### Qué Hace Esto

`AI_SUMMARIZE()` generates executive summaries:
- Input: All feedback from a specific journey stage
- Output: Concise summary highlighting key themes

### Use Case

Program managers need quick insights:
- "What are patients saying during Onboarding?"
- "What barriers appear during Refills?"
- "What's working well in Active Treatment?"

In [ ]:
CREATE OR REPLACE TABLE journey_stage_summaries AS
WITH stage_feedback AS (
    SELECT 
        journey_stage,
        LISTAGG(feedback_text, ' | ') WITHIN GROUP (ORDER BY feedback_id) as combined_feedback
    FROM (
        SELECT journey_stage, feedback_id, feedback_text
        FROM patient_feedback
        QUALIFY ROW_NUMBER() OVER (PARTITION BY journey_stage ORDER BY feedback_id) <= 20
    )
    GROUP BY journey_stage
)
SELECT
    journey_stage,
    SNOWFLAKE.CORTEX.SUMMARIZE(combined_feedback) as stage_summary,
    LENGTH(combined_feedback) as original_text_length,
    LENGTH(SNOWFLAKE.CORTEX.SUMMARIZE(combined_feedback)) as summary_length
FROM stage_feedback;

SELECT 
    journey_stage,
    stage_summary
FROM journey_stage_summaries
ORDER BY journey_stage;


---

## Paso 10: Classify Risk with CORTEX.CLASSIFY_TEXT()

### Qué Hace Esto

`AI_CLASSIFY()` performs zero-shot classification:
- Input: Patient feedback text + risk categories
- Output: JSON with predicted category and confidence score

### Risk Categories

We classify patients into:
- **Alto Riesgo** - Likely to drop off within 30 days
- **Riesgo Medio** - Some barriers, needs monitoring
- **Bajo Riesgo** - Engaged and progressing well
- **Optimal** - Altoly satisfied, on track

In [ ]:
CREATE OR REPLACE TABLE patient_risk_classification AS
WITH patient_recent_feedback AS (
    SELECT 
        pf.patient_id,
        LISTAGG(pf.feedback_text, '. ') WITHIN GROUP (ORDER BY pf.feedback_date DESC) as recent_feedback,
        COUNT(*) as feedback_count
    FROM patient_feedback pf
    GROUP BY pf.patient_id
    HAVING COUNT(*) >= 2
)
SELECT
    prf.patient_id,
    prf.recent_feedback,
    AI_CLASSIFY(
        prf.recent_feedback,
        ['High Risk', 'Medium Risk', 'Low Risk', 'Optimal']
    ) as risk_classification,
    -- Verified via Snowflake docs 2025-01: AI_CLASSIFY returns OBJECT, access fields directly
    risk_classification['labels'][0]::STRING as risk_category,
    1.0 as confidence_score  -- AI_CLASSIFY doesn't return confidence, use default
FROM patient_recent_feedback prf
LIMIT 300;

SELECT 
    risk_category,
    COUNT(*) as patient_count,
    ROUND(AVG(confidence_score), 3) as avg_confidence,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as percentage
FROM patient_risk_classification
GROUP BY risk_category
ORDER BY patient_count DESC;

---

## Paso 11: Interactive Journey Intelligence Dashboard

### Dashboard Features

This Streamlit dashboard integrates all 4 Cortex AI analyses:

**Tab 1: Journey Funnel**
- Completion rates by stage
- Drop-off point visualization
- Cohort comparison

**Tab 2: Sentiment Trends**
- Sentiment scores by journey stage
- Positive vs. negative distribution
- Alert triggers for declining sentiment

**Tab 3: Pain Point Analysis**
- Barrier category breakdown (from `CORTEX.COMPLETE`)
- Stage-specific issues
- Intervention priority matrix

**Tab 4: Risk Segmentation**
- Patient risk distribution (from `CORTEX.CLASSIFY_TEXT`)
- Alto-risk patient list
- Confidence scoring

In [ ]:
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🏥 Patient Journey Intelligence Dashboard")
st.markdown("### Powered by Snowflake Cortex AI")

# Data validation
try:
    data_check = session.sql("""
        SELECT 
            (SELECT COUNT(*) FROM patients) as patient_count,
            (SELECT COUNT(*) FROM journey_interactions) as interaction_count,
            (SELECT COUNT(*) FROM feedback_sentiment) as sentiment_count,
            (SELECT COUNT(*) FROM patient_risk_classification) as risk_count
    """).collect()[0]
    
    if data_check['PATIENT_COUNT'] == 0:
        st.error("❌ No data found! Please run **Cells 2-20** to generate data and analyses.")
        st.stop()
except Exception as e:
    st.error(f"❌ Required tables not found. Please run **Cells 2-20** first.")
    st.info(f"Error: {str(e)[:200]}")
    st.stop()

tab1, tab2, tab3, tab4 = st.tabs(["📊 Journey Funnel", "💭 Sentiment Trends", "⚠️ Pain Points", "🎯 Risk Segmentation"])

with tab1:
    st.subheader("Journey Completion Analysis")
    
    completion = session.sql("""
        SELECT completed_journey, COUNT(*) as count
        FROM journey_completion
        GROUP BY completed_journey
    """).to_pandas()
    
    col1, col2 = st.columns(2)
    with col1:
        completed = completion[completion['COMPLETED_JOURNEY'] == True]['COUNT'].sum() if True in completion['COMPLETED_JOURNEY'].values else 0
        total = completion['COUNT'].sum()
        st.metric("Completion Rate", f"{(completed/total*100):.1f}%", f"{completed} of {total} patients")
    with col2:
        dropped = completion[completion['COMPLETED_JOURNEY'] == False]['COUNT'].sum() if False in completion['COMPLETED_JOURNEY'].values else 0
        st.metric("Drop-Off Rate", f"{(dropped/total*100):.1f}%", f"{dropped} patients")
    
    dropoff_stages = session.sql("""
        SELECT drop_off_stage, COUNT(*) as count
        FROM journey_completion
        WHERE drop_off_stage IS NOT NULL
        GROUP BY drop_off_stage
        ORDER BY count DESC
    """).to_pandas()
    
    if len(dropoff_stages) > 0:
        chart = alt.Chart(dropoff_stages).mark_bar().encode(
            x=alt.X('COUNT:Q', title='Patient Count'),
            y=alt.Y('DROP_OFF_STAGE:N', title='Drop-Off Stage', sort='-x'),
            color=alt.value('#FF6B6B')
        ).properties(width=600, height=300, title='Drop-Off Points')
        st.altair_chart(chart, use_container_width=True)

with tab2:
    st.subheader("Sentiment Analysis by Journey Stage")
    
    sentiment_data = session.sql("""
        SELECT 
            journey_stage,
            sentiment_category,
            COUNT(*) as count,
            ROUND(AVG(sentiment_score), 3) as avg_score
        FROM feedback_sentiment
        GROUP BY journey_stage, sentiment_category
        ORDER BY journey_stage, sentiment_category
    """).to_pandas()
    
    if len(sentiment_data) > 0:
        chart = alt.Chart(sentiment_data).mark_bar().encode(
            x=alt.X('JOURNEY_STAGE:N', title='Journey Stage'),
            y=alt.Y('COUNT:Q', title='Feedback Count'),
            color=alt.Color('SENTIMENT_CATEGORY:N', title='Sentiment'),
            tooltip=['JOURNEY_STAGE', 'SENTIMENT_CATEGORY', 'COUNT', 'AVG_SCORE']
        ).properties(width=700, height=400, title='Sentiment Distribution by Stage')
        st.altair_chart(chart, use_container_width=True)
    
    st.markdown("#### Stage Summaries (CORTEX.SUMMARIZE)")
    summaries = session.sql("""
        SELECT journey_stage, stage_summary
        FROM journey_stage_summaries
        ORDER BY journey_stage
    """).to_pandas()
    
    for _, row in summaries.iterrows():
        with st.expander(f"📝 {row['JOURNEY_STAGE']}"):
            st.write(row['STAGE_SUMMARY'])

with tab3:
    st.subheader("Pain Point Analysis (CORTEX.COMPLETE)")
    
    barriers = session.sql("""
        SELECT 
            barrier_category,
            COUNT(*) as count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as percentage
        FROM pain_point_analysis
        GROUP BY barrier_category
        ORDER BY count DESC
    """).to_pandas()
    
    if len(barriers) > 0:
        chart = alt.Chart(barriers).mark_arc(innerRadius=50).encode(
            theta=alt.Theta('COUNT:Q'),
            color=alt.Color('BARRIER_CATEGORY:N', title='Barrier Type'),
            tooltip=['BARRIER_CATEGORY', 'COUNT', 'PERCENTAGE']
        ).properties(width=400, height=400, title='Barrier Distribution')
        st.altair_chart(chart, use_container_width=True)
        
        st.dataframe(barriers, use_container_width=True, hide_index=True)

with tab4:
    st.subheader("Patient Risk Classification (CORTEX.CLASSIFY_TEXT)")
    
    risk_dist = session.sql("""
        SELECT 
            risk_category,
            COUNT(*) as count,
            ROUND(AVG(confidence_score), 3) as avg_confidence
        FROM patient_risk_classification
        GROUP BY risk_category
        ORDER BY count DESC
    """).to_pandas()
    
    if len(risk_dist) > 0:
        cols = st.columns(len(risk_dist))
        for idx, row in risk_dist.iterrows():
            with cols[idx]:
                st.metric(row['RISK_CATEGORY'], f"{row['COUNT']} patients", f"{row['AVG_CONFIDENCE']:.2f} conf")
        
        chart = alt.Chart(risk_dist).mark_bar().encode(
            x=alt.X('RISK_CATEGORY:N', title='Risk Category'),
            y=alt.Y('COUNT:Q', title='Patient Count'),
            color=alt.Color('RISK_CATEGORY:N', 
                scale=alt.Scale(domain=['High Risk', 'Medium Risk', 'Low Risk', 'Optimal'],
                               range=['#FF4444', '#FFA500', '#FFD700', '#4CAF50']))
        ).properties(width=600, height=400, title='Risk Distribution')
        st.altair_chart(chart, use_container_width=True)
        
        high_risk = session.sql("""
            SELECT patient_id, recent_feedback, confidence_score
            FROM patient_risk_classification
            WHERE risk_category = 'High Risk'
            ORDER BY confidence_score DESC
            LIMIT 10
        """).to_pandas()
        
        if len(high_risk) > 0:
            st.markdown("#### 🚨 High-Risk Patients (Immediate Intervention)")
            st.dataframe(high_risk, use_container_width=True, hide_index=True)

st.success("✅ Dashboard operational | All 4 Cortex AI functions integrated")


---

## Completion Summary

### What You've Built

A complete **patient journey intelligence system** using Snowflake Cortex AI:

 **Sentiment tracking** across 6 journey stages with `CORTEX.SENTIMENT()`
 **Pain point extraction** from negative feedback with `CORTEX.COMPLETE()`
 **Executive summaries** by journey stage with `CORTEX.SUMMARIZE()`
 **Risk classification** for proactive intervention with `CORTEX.CLASSIFY_TEXT()`
 **Interactive dashboard** with 4 analytical views

### Key Metrics Generated

- **1,000 patients** across 3 cohorts
- **6,000+ interactions** with 5 touchpoint types
- **3,000+ feedback entries** with sentiment analysis
- **Completion tracking** with drop-off detection

### Impacto de Negocio

This system enables:
1. **Proactive patient support** - Identify at-risk patients before they drop off
2. **Data-driven interventions** - Target specific barriers with appropriate programs
3. **Journey optimization** - Improve stages with highest drop-off rates
4. **Resource efficiency** - Focus support efforts on high-impact areas

### Next Steps

1. **Customize journey stages** for your specific treatment protocols
2. **Integrate real patient data** (feedback, EHR, CRM)
3. **Build intervention workflows** based on risk scores
4. **Set up automated alerts** for declining sentiment trends
5. **Expand Cortex AI usage** (TRANSLATE for multilingual, EXTRACT_ANSWER for Q&A)

---

 **Congratulations!** You've successfully built an AI-powered patient journey intelligence system.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `PATIENT_JOURNEY_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and analyses will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS PATIENT_JOURNEY_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
